In [ ]:
!nvidia-smi

Wed Jul  8 15:11:49 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   53C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
!pip install -q -U datasets transformers evaluate seqeval accelerate

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 3.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 19.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.5/11.5 MB 58.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 17.7 MB/s eta 0:00:00


In [ ]:
import torch
print("torch", torch.__version__, "| CUDA:", torch.cuda.is_available())

from google.colab import drive
drive.mount("/content/drive")

import json, os, gc, math
import numpy as np
import pandas as pd
from datasets import load_from_disk
from transformers import (AutoTokenizer, AutoModelForTokenClassification,
                          TrainingArguments, Trainer,
                          DataCollatorForTokenClassification)
from seqeval.metrics import (classification_report, f1_score,
                             precision_score, recall_score)

BASE = "/content/drive/MyDrive/ner"
splits = load_from_disk(f"{BASE}/uzbek_ner_splits")          # tokens + tags
label_list = json.load(open(f"{BASE}/label_list.json"))
label2id = {l: i for i, l in enumerate(label_list)}
id2label = {i: l for l, i in label2id.items()}
CLASSES = ["PERSON", "ORG", "GPE", "LOC", "DATE"]

print(splits)

torch 2.11.0+cu128 | CUDA: True
Mounted at /content/drive
DatasetDict({
    train: Dataset({
        features: ['tokens', 'tags'],
        num_rows: 15687
    })
    validation: Dataset({
        features: ['tokens', 'tags'],
        num_rows: 1961
    })
    test: Dataset({
        features: ['tokens', 'tags'],
        num_rows: 1961
    })
})


In [ ]:
# ---- entity-level metric (direct seqeval — no Hub call, offline-friendly) ----
def compute_metrics(p):
    logits, labels = p
    preds = np.argmax(logits, axis=2)
    tp, tl = [], []
    for pr, la in zip(preds, labels):
        tp.append([id2label[int(q)] for q, l in zip(pr, la) if l != -100])
        tl.append([id2label[int(l)] for q, l in zip(pr, la) if l != -100])
    return {"precision": precision_score(tl, tp),
            "recall":    recall_score(tl, tp),
            "f1":        f1_score(tl, tp)}


# ---- the four evaluation views from the baseline decomposition ----
def merge_place(seqs):
    return [[t.replace("-GPE", "-PLACE").replace("-LOC", "-PLACE") for t in s]
            for s in seqs]

def drop_date(seqs):
    return [["O" if t.endswith("DATE") else t for t in s] for s in seqs]

def four_views(gold, pred):
    return {
        "f1_5class": round(f1_score(gold, pred), 4),
        "f1_place":  round(f1_score(merge_place(gold), merge_place(pred)), 4),
        "f1_nodate": round(f1_score(drop_date(gold), drop_date(pred)), 4),
        "f1_core":   round(f1_score(merge_place(drop_date(gold)),
                                    merge_place(drop_date(pred))), 4),
    }


# ---- truncation audit, per tokenizer (Phase 4's logic, automated) ----
def pick_maxlen(tokenizer, split, candidates=(192, 256, 320, 384)):
    span_first = []
    for ex in split:
        enc = tokenizer(ex["tokens"], is_split_into_words=True)
        wid = enc.word_ids()
        first = {}
        for pos, w in enumerate(wid):
            if w is not None and w not in first:
                first[w] = pos
        for i, t in enumerate(ex["tags"]):
            if t.startswith("B-") and i in first:
                span_first.append(first[i])
    sp = np.array(span_first)
    for L in candidates:
        lost = int((sp >= L - 1).sum())
        print(f"    max_length={L}: {lost} spans lost")
        if lost == 0:
            return L
    return candidates[-1]


# ---- label alignment, parameterized by tokenizer ----
def make_align_fn(tokenizer, maxlen):
    def fn(batch):
        enc = tokenizer(batch["tokens"], is_split_into_words=True,
                        truncation=True, max_length=maxlen)
        all_labels = []
        for i, tags in enumerate(batch["tags"]):
            wid = enc.word_ids(batch_index=i)
            prev, row = None, []
            for w in wid:
                if w is None:
                    row.append(-100)
                elif w != prev:
                    row.append(label2id[tags[w]])
                else:
                    row.append(-100)
                prev = w
            all_labels.append(row)
        enc["labels"] = all_labels
        return enc
    return fn

In [ ]:
def run_experiment(model_name, run_tag, batch=16, accum=1, epochs=4, lr=2e-5, seed=42):
    print("=" * 62)
    print("RUN:", run_tag, "|", model_name)

    tokenizer = AutoTokenizer.from_pretrained(model_name)
    assert tokenizer.is_fast, "slow tokenizer: no word_ids -> cannot align (see Step 4 note)"

    print("  auditing max_length for this tokenizer:")
    maxlen = pick_maxlen(tokenizer, splits["train"])
    print("  chosen max_length:", maxlen)

    tok_data = splits.map(make_align_fn(tokenizer, maxlen), batched=True,
                          remove_columns=["tokens", "tags"])

    model = AutoModelForTokenClassification.from_pretrained(
        model_name, num_labels=len(label_list),
        id2label=id2label, label2id=label2id)

    steps_per_epoch = math.ceil(len(tok_data["train"]) / (batch * accum))
    args = TrainingArguments(
        output_dir=f"run-{run_tag}",
        eval_strategy="epoch", save_strategy="epoch",
        learning_rate=lr,
        per_device_train_batch_size=batch,
        gradient_accumulation_steps=accum,
        per_device_eval_batch_size=32,
        num_train_epochs=epochs,
        weight_decay=0.01,
        warmup_steps=int(0.06 * steps_per_epoch * epochs),
        fp16=True,
        load_best_model_at_end=True, metric_for_best_model="f1",
        greater_is_better=True,
        save_total_limit=1, logging_steps=50, seed=seed,
        report_to="none",
    )
    trainer = Trainer(model=model, args=args,
                      train_dataset=tok_data["train"],
                      eval_dataset=tok_data["validation"],
                      data_collator=DataCollatorForTokenClassification(tokenizer),
                      processing_class=tokenizer,
                      compute_metrics=compute_metrics)

    trainer.train()

    # validation -> word-level sequences (the familiar -100 strip)
    pred = trainer.predict(tok_data["validation"])
    preds = np.argmax(pred.predictions, axis=2)
    labs = pred.label_ids
    p_seqs, g_seqs = [], []
    for pr, la in zip(preds, labs):
        p_seqs.append([id2label[int(q)] for q, l in zip(pr, la) if l != -100])
        g_seqs.append([id2label[int(l)] for q, l in zip(pr, la) if l != -100])

    print(classification_report(g_seqs, p_seqs, digits=3))
    views = four_views(g_seqs, p_seqs)
    print("four views:", views)

    # keep the model for the (single, later) test run
    trainer.save_model(f"models/{run_tag}")
    tokenizer.save_pretrained(f"models/{run_tag}")

    # append the ledger row on Drive
    row = {"run": run_tag, "model": model_name, "max_length": maxlen,
           "lr": lr, "epochs": epochs, "eff_batch": batch * accum, "seed": seed,
           "val_precision": round(precision_score(g_seqs, p_seqs), 4),
           "val_recall":    round(recall_score(g_seqs, p_seqs), 4),
           **views}
    path = f"{BASE}/runs_ledger.csv"
    df = pd.read_csv(path) if os.path.exists(path) else pd.DataFrame()
    df = pd.concat([df, pd.DataFrame([row])], ignore_index=True)
    df.to_csv(path, index=False)
    print("ledger updated ->", path)

    return trainer, g_seqs, p_seqs

In [ ]:
trainer, g_seqs, p_seqs = run_experiment("bert-base-multilingual-cased", run_tag="mbert")

RUN: mbert | bert-base-multilingual-cased


config.json:   0%|          | 0.00/625 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/996k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.96M [00:00<?, ?B/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.atte

              precision    recall  f1-score   support

        DATE      0.409     0.531     0.462      1035
         GPE      0.714     0.824     0.765      4931
         LOC      0.500     0.455     0.477      2144
         ORG      0.616     0.699     0.655      3267
      PERSON      0.867     0.871     0.869      3041

   micro avg      0.668     0.730     0.698     14418
   macro avg      0.621     0.676     0.646     14418
weighted avg      0.670     0.730     0.697     14418

four views: {'f1_5class': np.float64(0.6976), 'f1_place': np.float64(0.7584), 'f1_nodate': np.float64(0.7177), 'f1_core': np.float64(0.7838)}


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

ledger updated -> /content/drive/MyDrive/ner/runs_ledger.csv


In [ ]:
del trainer
gc.collect()
torch.cuda.empty_cache()

In [ ]:
t = AutoTokenizer.from_pretrained("tahrirchi/tahrirchi-bert-base")
print("fast:", t.is_fast)

demo = t("O'zbekiston Respublikasi sudlari to'g'risida".split(),
         is_split_into_words=True)
print(t.convert_ids_to_tokens(demo["input_ids"]))
print(demo.word_ids())

config.json:   0%|          | 0.00/676 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.22k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/491k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/287k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.30M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

fast: True
['<s>', 'O', "'", 'zbekiston', 'Respublika', 'si', 'sud', 'lari', 'to', "'", 'g', "'", 'risida', '</s>']
[None, 0, 0, 0, 1, 1, 2, 2, 3, 3, 3, 3, 3, None]


In [ ]:
trainer, g_seqs, p_seqs = run_experiment(
    "tahrirchi/tahrirchi-bert-base", run_tag="tahrirchi")

RUN: tahrirchi | tahrirchi/tahrirchi-bert-base
  auditing max_length for this tokenizer:
    max_length=192: 12969 spans lost
    max_length=256: 1695 spans lost
    max_length=320: 162 spans lost
    max_length=384: 5 spans lost
  chosen max_length: 384


Map:   0%|          | 0/15687 [00:00<?, ? examples/s]

Map:   0%|          | 0/1961 [00:00<?, ? examples/s]

Map:   0%|          | 0/1961 [00:00<?, ? examples/s]

model.safetensors:   0%|          | 0.00/438M [00:00<?, ?B/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

              precision    recall  f1-score   support

        DATE      0.396     0.523     0.450      1035
         GPE      0.708     0.821     0.760      4931
         LOC      0.487     0.439     0.462      2144
         ORG      0.587     0.659     0.621      3267
      PERSON      0.846     0.856     0.851      3043

   micro avg      0.652     0.713     0.682     14420
   macro avg      0.605     0.660     0.629     14420
weighted avg      0.654     0.713     0.681     14420

four views: {'f1_5class': np.float64(0.6816), 'f1_place': np.float64(0.7434), 'f1_nodate': np.float64(0.7015), 'f1_core': np.float64(0.7688)}


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

ledger updated -> /content/drive/MyDrive/ner/runs_ledger.csv


In [ ]:
del trainer
gc.collect()
torch.cuda.empty_cache()

In [ ]:
trainer, g_seqs, p_seqs = run_experiment(
    "xlm-roberta-large", run_tag="xlmr-large", batch=8, accum=2)

RUN: xlmr-large | xlm-roberta-large


config.json:   0%|          | 0.00/616 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.10M [00:00<?, ?B/s]

...

Epoch,Training Loss,Validation Loss,Precision,Recall,F1
1,0.320605,0.155087,0.670812,0.695839,0.683096
2,0.277384,0.145196,0.677938,0.731345,0.703630
3,0.235836,0.144674,0.694441,0.743273,0.718028
4,0.201933,0.152470,0.704188,0.746186,0.724579


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

              precision    recall  f1-score   support

        DATE      0.426     0.500     0.460      1035
         GPE      0.740     0.837     0.785      4931
         LOC      0.543     0.503     0.522      2144
         ORG      0.686     0.718     0.702      3267
      PERSON      0.873     0.884     0.879      3043

   micro avg      0.704     0.746     0.725     14420
   macro avg      0.654     0.689     0.670     14420
weighted avg      0.704     0.746     0.724     14420

four views: {'f1_5class': np.float64(0.7246), 'f1_place': np.float64(0.7825), 'f1_nodate': np.float64(0.7463), 'f1_core': np.float64(0.8089)}


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

ledger updated -> /content/drive/MyDrive/ner/runs_ledger.csv


In [ ]:
WINNER = "xlmr-large"

tokenizer = AutoTokenizer.from_pretrained(f"models/{WINNER}")
tok_test = splits.map(make_align_fn(tokenizer, 384),
                      batched=True, remove_columns=["tokens", "tags"])
model = AutoModelForTokenClassification.from_pretrained(f"models/{WINNER}")

eval_args = TrainingArguments(output_dir="final-test",
                              per_device_eval_batch_size=32,
                              fp16=True, report_to="none")
tester = Trainer(model=model, args=eval_args,
                 data_collator=DataCollatorForTokenClassification(tokenizer),
                 processing_class=tokenizer, compute_metrics=compute_metrics)

pred = tester.predict(tok_test["test"])
preds = np.argmax(pred.predictions, axis=2)
labs = pred.label_ids
tp_seqs, tg_seqs = [], []
for pr, la in zip(preds, labs):
    tp_seqs.append([id2label[int(q)] for q, l in zip(pr, la) if l != -100])
    tg_seqs.append([id2label[int(l)] for q, l in zip(pr, la) if l != -100])

print(classification_report(tg_seqs, tp_seqs, digits=3))
print("TEST four views:", four_views(tg_seqs, tp_seqs))

Map:   0%|          | 0/15687 [00:00<?, ? examples/s]

Map:   0%|          | 0/1961 [00:00<?, ? examples/s]

Map:   0%|          | 0/1961 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

              precision    recall  f1-score   support

        DATE      0.458     0.506     0.481      1048
         GPE      0.743     0.816     0.777      5096
         LOC      0.522     0.506     0.514      2120
         ORG      0.646     0.704     0.674      2988
      PERSON      0.861     0.876     0.869      2945

   micro avg      0.693     0.736     0.714     14197
   macro avg      0.646     0.682     0.663     14197
weighted avg      0.693     0.736     0.713     14197

TEST four views: {'f1_5class': np.float64(0.7139), 'f1_place': np.float64(0.7751), 'f1_nodate': np.float64(0.7329), 'f1_core': np.float64(0.7991)}


In [ ]:
import shutil
shutil.copytree(f"models/{WINNER}", f"{BASE}/final_model_{WINNER}", dirs_exist_ok=True)
print("winner safe on Drive")

winner safe on Drive


In [ ]:
ledger = pd.read_csv(f"{BASE}/runs_ledger.csv")

ledger["place_tax"] = (ledger["f1_place"] - ledger["f1_5class"]).round(4)
ledger["date_drag"] = (ledger["f1_nodate"] - ledger["f1_5class"]).round(4)

cols = ["run", "model", "max_length", "val_precision", "val_recall",
        "f1_5class", "f1_place", "f1_nodate", "f1_core",
        "place_tax", "date_drag"]
print(ledger[cols].to_string(index=False))

       run                         model  max_length  val_precision  val_recall  f1_5class  f1_place  f1_nodate  f1_core  place_tax  date_drag
  baseline              xlm-roberta-base         256         0.6750      0.7400     0.7060    0.7699     0.7277   0.7970     0.0639     0.0217
     mbert  bert-base-multilingual-cased         384         0.6680      0.7299     0.6976    0.7584     0.7177   0.7838     0.0608     0.0201
 tahrirchi tahrirchi/tahrirchi-bert-base         384         0.6524      0.7135     0.6816    0.7434     0.7015   0.7688     0.0618     0.0199
xlmr-large             xlm-roberta-large         384         0.7042      0.7462     0.7246    0.7825     0.7463   0.8089     0.0579     0.0217


In [ ]:
import os

WINNER = "xlmr-large"
wrow = ledger[ledger["run"] == WINNER].iloc[-1]
MODEL_DIR = f"models/{WINNER}"

print("model on local disk:", os.path.isdir(MODEL_DIR))
print("final model:", wrow["model"], "| val f1_5class:", wrow["f1_5class"],
      "| max_length:", int(wrow["max_length"]))

model on local disk: True
final model: xlm-roberta-large | val f1_5class: 0.7246 | max_length: 384


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)
tok_test = splits.map(make_align_fn(tokenizer, int(wrow["max_length"])),
                      batched=True, remove_columns=["tokens", "tags"])
model = AutoModelForTokenClassification.from_pretrained(MODEL_DIR)

eval_args = TrainingArguments(output_dir="final-test",
                              per_device_eval_batch_size=32,
                              fp16=True, report_to="none")
tester = Trainer(model=model, args=eval_args,
                 data_collator=DataCollatorForTokenClassification(tokenizer),
                 processing_class=tokenizer, compute_metrics=compute_metrics)

pred = tester.predict(tok_test["test"])
preds = np.argmax(pred.predictions, axis=2)
labs = pred.label_ids

tp_seqs, tg_seqs = [], []
for pr, la in zip(preds, labs):
    tp_seqs.append([id2label[int(q)] for q, l in zip(pr, la) if l != -100])
    tg_seqs.append([id2label[int(l)] for q, l in zip(pr, la) if l != -100])

print(classification_report(tg_seqs, tp_seqs, digits=3))
print("TEST four views:", four_views(tg_seqs, tp_seqs))

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

              precision    recall  f1-score   support

        DATE      0.458     0.506     0.481      1048
         GPE      0.743     0.816     0.777      5096
         LOC      0.522     0.506     0.514      2120
         ORG      0.646     0.704     0.674      2988
      PERSON      0.861     0.876     0.869      2945

   micro avg      0.693     0.736     0.714     14197
   macro avg      0.646     0.682     0.663     14197
weighted avg      0.693     0.736     0.713     14197

TEST four views: {'f1_5class': np.float64(0.7139), 'f1_place': np.float64(0.7751), 'f1_nodate': np.float64(0.7329), 'f1_core': np.float64(0.7991)}


In [ ]:
import shutil, json

# the champion, safe on Drive (~2.2 GB — skip if already copied)
shutil.copytree(MODEL_DIR, f"{BASE}/final_model_{WINNER}", dirs_exist_ok=True)

# the test results as files, so Phase 8 doesn't depend on screenshots
with open(f"{BASE}/final_test_report.txt", "w") as f:
    f.write(classification_report(tg_seqs, tp_seqs, digits=3))
json.dump({k: float(v) for k, v in four_views(tg_seqs, tp_seqs).items()},
          open(f"{BASE}/final_test_views.json", "w"), indent=2)

print("saved: final model + test report + views -> Drive")

saved: final model + test report + views -> Drive


In [ ]:
PHASE 8 STARTS FROM HERE

In [ ]:
import torch
print("torch", torch.__version__, "| CUDA:", torch.cuda.is_available())

from google.colab import drive
drive.mount("/content/drive")

import json, os
import numpy as np
from collections import Counter
from datasets import load_from_disk
from transformers import AutoTokenizer

BASE = "/content/drive/MyDrive/ner"
splits = load_from_disk(f"{BASE}/uzbek_ner_splits")
label_list = json.load(open(f"{BASE}/label_list.json"))
CLASSES = ["PERSON", "ORG", "GPE", "LOC", "DATE"]
print(splits)

print()
print("=== artifact inventory ===")
for name in ["final_model_xlmr-large", "baseline_xlmr", "uzbek_ner_splits",
             "uzbek_ner_tokenized", "label_list.json", "runs_ledger.csv",
             "final_test_report.txt", "final_test_views.json"]:
    print(f"{name:26}", "OK" if os.path.exists(f"{BASE}/{name}") else "MISSING")

torch 2.11.0+cu128 | CUDA: True
Mounted at /content/drive
DatasetDict({
    train: Dataset({
        features: ['tokens', 'tags'],
        num_rows: 15687
    })
    validation: Dataset({
        features: ['tokens', 'tags'],
        num_rows: 1961
    })
    test: Dataset({
        features: ['tokens', 'tags'],
        num_rows: 1961
    })
})

=== artifact inventory ===
final_model_xlmr-large     OK
baseline_xlmr              OK
uzbek_ner_splits           OK
uzbek_ner_tokenized        OK
label_list.json            OK
runs_ledger.csv            OK
final_test_report.txt      OK
final_test_views.json      OK


In [ ]:
def b_share(split):
    c, tot = Counter(), 0
    for tags in split["tags"]:
        c.update(tags)
        tot += len(tags)
    return {cl: round(c["B-" + cl] / tot * 100, 2) for cl in CLASSES}

for name in splits:
    print(f"{name:11}", b_share(splits[name]))

train       {'PERSON': 1.73, 'ORG': 1.82, 'GPE': 2.93, 'LOC': 1.27, 'DATE': 0.65}
validation  {'PERSON': 1.78, 'ORG': 1.91, 'GPE': 2.89, 'LOC': 1.26, 'DATE': 0.61}
test        {'PERSON': 1.7, 'ORG': 1.73, 'GPE': 2.95, 'LOC': 1.23, 'DATE': 0.61}


In [ ]:
tok = AutoTokenizer.from_pretrained(f"{BASE}/baseline_xlmr")

def tok_len(batch):
    enc = tok(batch["tokens"], is_split_into_words=True)
    return {"n_tok": [len(x) for x in enc["input_ids"]]}

lens = np.array(splits["train"].map(tok_len, batched=True,
                remove_columns=splits["train"].column_names)["n_tok"])
words = np.array([len(t) for t in splits["train"]["tokens"]])

print("tokens/row: min", lens.min(),
      " p50/p90/p99/p99.5", np.percentile(lens, [50, 90, 99, 99.5]).astype(int),
      " max", lens.max())
print("pieces per word (median):",
      round(np.percentile(lens, 50) / np.percentile(words, 50), 2))

Map:   0%|          | 0/15687 [00:00<?, ? examples/s]

tokens/row: min 28  p50/p90/p99/p99.5 [196 253 289 303]  max 361
pieces per word (median): 2.2


In [ ]:
from transformers import (AutoModelForTokenClassification, TrainingArguments,
                          Trainer, DataCollatorForTokenClassification)
from seqeval.metrics import classification_report, f1_score

def make_align_fn(tokenizer, maxlen):
    def fn(batch):
        enc = tokenizer(batch["tokens"], is_split_into_words=True,
                        truncation=True, max_length=maxlen)
        all_labels = []
        for i, tags in enumerate(batch["tags"]):
            wid = enc.word_ids(batch_index=i)
            prev, row = None, []
            for w in wid:
                if w is None:
                    row.append(-100)
                elif w != prev:
                    row.append({l: i2 for i2, l in enumerate(label_list)}[tags[w]] if False else label_list.index(tags[w]))
                else:
                    row.append(-100)
                prev = w
            all_labels.append(row)
        enc["labels"] = all_labels
        return enc
    return fn

label2id = {l: i for i, l in enumerate(label_list)}
id2label = {i: l for l, i in label2id.items()}

def merge_place(seqs):
    return [[t.replace("-GPE", "-PLACE").replace("-LOC", "-PLACE") for t in s] for s in seqs]
def drop_date(seqs):
    return [["O" if t.endswith("DATE") else t for t in s] for s in seqs]
def four_views(gold, pred):
    return {"f1_5class": round(f1_score(gold, pred), 4),
            "f1_place":  round(f1_score(merge_place(gold), merge_place(pred)), 4),
            "f1_nodate": round(f1_score(drop_date(gold), drop_date(pred)), 4),
            "f1_core":   round(f1_score(merge_place(drop_date(gold)),
                                        merge_place(drop_date(pred))), 4)}

MODEL_DIR = f"{BASE}/final_model_xlmr-large"
tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)
model = AutoModelForTokenClassification.from_pretrained(MODEL_DIR)

def align(batch):
    enc = tokenizer(batch["tokens"], is_split_into_words=True,
                    truncation=True, max_length=384)
    all_labels = []
    for i, tags in enumerate(batch["tags"]):
        wid = enc.word_ids(batch_index=i)
        prev, row = None, []
        for w in wid:
            if w is None: row.append(-100)
            elif w != prev: row.append(label2id[tags[w]])
            else: row.append(-100)
            prev = w
        all_labels.append(row)
    enc["labels"] = all_labels
    return enc

tok_test = splits.map(align, batched=True, remove_columns=["tokens", "tags"])
tester = Trainer(model=model,
                 args=TrainingArguments(output_dir="final-test",
                                        per_device_eval_batch_size=32,
                                        fp16=True, report_to="none"),
                 data_collator=DataCollatorForTokenClassification(tokenizer),
                 processing_class=tokenizer)

pred = tester.predict(tok_test["test"])
preds = np.argmax(pred.predictions, axis=2)
labs = pred.label_ids
tp_seqs, tg_seqs = [], []
for pr, la in zip(preds, labs):
    tp_seqs.append([id2label[int(q)] for q, l in zip(pr, la) if l != -100])
    tg_seqs.append([id2label[int(l)] for q, l in zip(pr, la) if l != -100])

views = four_views(tg_seqs, tp_seqs)
print("reproduced test views:", views)
assert abs(views["f1_5class"] - 0.7139) < 0.0005, "test F1 did not reproduce!"

with open(f"{BASE}/final_test_report.txt", "w") as f:
    f.write(classification_report(tg_seqs, tp_seqs, digits=3))
json.dump({k: float(v) for k, v in views.items()},
          open(f"{BASE}/final_test_views.json", "w"), indent=2)
print("artifacts written to Drive: final_test_report.txt, final_test_views.json")

Loading weights:   0%|          | 0/391 [00:05<?, ?it/s]

Map:   0%|          | 0/15687 [00:00<?, ? examples/s]

Map:   0%|          | 0/1961 [00:00<?, ? examples/s]

Map:   0%|          | 0/1961 [00:00<?, ? examples/s]

reproduced test views: {'f1_5class': np.float64(0.7139), 'f1_place': np.float64(0.7751), 'f1_nodate': np.float64(0.7329), 'f1_core': np.float64(0.7991)}
artifacts written to Drive: final_test_report.txt, final_test_views.json


In [ ]:
from transformers import pipeline

ner = pipeline("token-classification", model=MODEL_DIR,
               aggregation_strategy="simple",
               device=0 if torch.cuda.is_available() else -1)

MASK = {"PERSON": "[SHAXS]", "GPE": "[JOY]", "LOC": "[JOY]",
        "ORG": "[TASHKILOT]", "DATE": "[SANA]"}

def depersonalize(text):
    out = text
    for e in sorted(ner(text), key=lambda x: x["start"], reverse=True):
        m = MASK.get(e["entity_group"])
        if m:
            out = out[:e["start"]] + m + out[e["end"]:]
    return out

sentence = ("Toshkent shahar sudi 2026-yil 15-yanvar kuni Karimov Aziz va "
            "«Agrobank» ATB o'rtasidagi fuqarolik ishini ko'rib chiqdi.")

print("KIRISH: ", sentence)
for e in ner(sentence):
    print(f"  {e['entity_group']:7} {e['word']!r}  (score {e['score']:.2f})")
print("CHIQISH:", depersonalize(sentence))

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

KIRISH:  Toshkent shahar sudi 2026-yil 15-yanvar kuni Karimov Aziz va «Agrobank» ATB o'rtasidagi fuqarolik ishini ko'rib chiqdi.
  LOC     'Toshkent'  (score 0.71)
  DATE    '20'  (score 0.62)
  ORG     '«'  (score 1.00)
  ORG     'Agrobank»'  (score 0.86)
CHIQISH: [JOY] shahar sudi [SANA]26-yil 15-yanvar kuni Karimov Aziz va [TASHKILOT][TASHKILOT] ATB o'rtasidagi fuqarolik ishini ko'rib chiqdi.


In [ ]:
from transformers import pipeline

ner = pipeline("token-classification", model=MODEL_DIR,
               aggregation_strategy="first",
               device=0 if torch.cuda.is_available() else -1)

MASK = {"PERSON": "[SHAXS]", "GPE": "[JOY]", "LOC": "[JOY]",
        "ORG": "[TASHKILOT]", "DATE": "[SANA]"}

def depersonalize(text):
    out = text
    for e in sorted(ner(text), key=lambda x: x["start"], reverse=True):
        m = MASK.get(e["entity_group"])
        if m:
            out = out[:e["start"]] + m + out[e["end"]:]
    return out

sentences = [
    # A — control: fully in-distribution (news register, famous name)
    ("O'zbekiston Prezidenti Shavkat Mirziyoyev 2026-yil 15-yanvar kuni "
     "Toshkentda «Agrobank» rahbari bilan uchrashdi."),
    # B — legal register, but name in the news-typical Name-Surname order
    ("Toshkent shahar sudi 2026-yil 15-yanvar kuni Aziz Karimov va "
     "«Agrobank» aksiyadorlik tijorat banki o'rtasidagi fuqarolik ishini ko'rib chiqdi."),
    # C — legal register, official Surname-Name order (the hard case)
    ("Toshkent shahar sudi 2026-yil 15-yanvar kuni Karimov Aziz va "
     "«Agrobank» ATB o'rtasidagi fuqarolik ishini ko'rib chiqdi."),
]

for tag, s in zip("ABC", sentences):
    print(f"===== sentence {tag} =====")
    print("KIRISH: ", s)
    for e in ner(s):
        print(f"  {e['entity_group']:7} {e['word']!r}  (score {e['score']:.2f})")
    print("CHIQISH:", depersonalize(s))
    print()

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

===== sentence A =====
KIRISH:  O'zbekiston Prezidenti Shavkat Mirziyoyev 2026-yil 15-yanvar kuni Toshkentda «Agrobank» rahbari bilan uchrashdi.
  GPE     "O'zbekiston"  (score 1.00)
  PERSON  'ShavkatMirziyoyev'  (score 0.97)
  GPE     'Toshkentda'  (score 0.59)
  ORG     '«Agrobank»'  (score 1.00)
CHIQISH: [JOY] Prezidenti [SHAXS] 2026-yil 15-yanvar kuni [JOY] [TASHKILOT] rahbari bilan uchrashdi.

===== sentence B =====
KIRISH:  Toshkent shahar sudi 2026-yil 15-yanvar kuni Aziz Karimov va «Agrobank» aksiyadorlik tijorat banki o'rtasidagi fuqarolik ishini ko'rib chiqdi.
  LOC     'Toshkent'  (score 0.60)
  PERSON  'AzizKarimov'  (score 0.80)
  ORG     '«Agrobank»'  (score 1.00)
CHIQISH: [JOY] shahar sudi 2026-yil 15-yanvar kuni [SHAXS] va [TASHKILOT] aksiyadorlik tijorat banki o'rtasidagi fuqarolik ishini ko'rib chiqdi.

===== sentence C =====
KIRISH:  Toshkent shahar sudi 2026-yil 15-yanvar kuni Karimov Aziz va «Agrobank» ATB o'rtasidagi fuqarolik ishini ko'rib chiqdi.
  LOC     'Tos

/usr/local/lib/python3.12/dist-packages/transformers/pipelines/token_classification.py:444: UserWarning: Tokenizer does not support real words, using fallback heuristic
  warnings.warn(


In [ ]:
sentences = [
    # B2 — legal register, news-order name, TRAINING-FORMAT date (spaced)
    ("Toshkent shahar sudi 2026 yil 15 yanvar kuni Aziz Karimov va "
     "«Agrobank» aksiyadorlik tijorat banki o'rtasidagi fuqarolik ishini ko'rib chiqdi."),
    # C — unchanged: the documented hard case (surname-first official order)
    ("Toshkent shahar sudi 2026-yil 15-yanvar kuni Karimov Aziz va "
     "«Agrobank» ATB o'rtasidagi fuqarolik ishini ko'rib chiqdi."),
]

for tag, s in zip(["B2", "C"], sentences):
    print(f"===== sentence {tag} =====")
    print("KIRISH: ", s)
    for e in ner(s):
        print(f"  {e['entity_group']:7} {e['word']!r}  (score {e['score']:.2f})")
    print("CHIQISH:", depersonalize(s))
    print()

===== sentence B2 =====
KIRISH:  Toshkent shahar sudi 2026 yil 15 yanvar kuni Aziz Karimov va «Agrobank» aksiyadorlik tijorat banki o'rtasidagi fuqarolik ishini ko'rib chiqdi.
  LOC     'Toshkent'  (score 0.59)
  PERSON  'AzizKarimov'  (score 0.87)
  ORG     '«Agrobank»'  (score 1.00)
CHIQISH: [JOY] shahar sudi 2026 yil 15 yanvar kuni [SHAXS] va [TASHKILOT] aksiyadorlik tijorat banki o'rtasidagi fuqarolik ishini ko'rib chiqdi.

===== sentence C =====
KIRISH:  Toshkent shahar sudi 2026-yil 15-yanvar kuni Karimov Aziz va «Agrobank» ATB o'rtasidagi fuqarolik ishini ko'rib chiqdi.
  LOC     'Toshkent'  (score 0.71)
  DATE    '2026-yil'  (score 0.62)
  ORG     '«Agrobank»'  (score 1.00)
CHIQISH: [JOY] shahar sudi [SANA] 15-yanvar kuni Karimov Aziz va [TASHKILOT] ATB o'rtasidagi fuqarolik ishini ko'rib chiqdi.



In [10]:
import os
for f in ["phase1_category_frequency.csv", "phase2_conversion_report.csv"]:
    print(f, "->", "on Drive" if os.path.exists(f"{BASE}/{f}") else "MISSING")

phase1_category_frequency.csv -> MISSING
phase2_conversion_report.csv -> MISSING


In [11]:
import pandas as pd

rep = pd.DataFrame({
    "listed":       {"PERSON": 25006, "ORG": 27306, "GPE": 50031, "LOC": 38647, "DATE": 10901},
    "matched":      {"PERSON": 24453, "ORG": 22299, "GPE": 31228, "LOC": 16404, "DATE": 9874},
    "spans_tagged": {"PERSON": 29747, "ORG": 31187, "GPE": 50108, "LOC": 21670, "DATE": 11011},
})
rep["match_rate_%"] = (rep["matched"] / rep["listed"] * 100).round(1)
rep = rep[["listed", "matched", "match_rate_%", "spans_tagged"]].loc[
       ["PERSON", "ORG", "GPE", "LOC", "DATE"]]
rep.to_csv(f"{BASE}/phase2_conversion_report.csv")
print(rep.to_string()); print("\nwritten to Drive")

        listed  matched  match_rate_%  spans_tagged
PERSON   25006    24453          97.8         29747
ORG      27306    22299          81.7         31187
GPE      50031    31228          62.4         50108
LOC      38647    16404          42.4         21670
DATE     10901     9874          90.6         11011

written to Drive


In [12]:
from collections import Counter
from datasets import load_dataset
import pandas as pd

raw = load_dataset("risqaliyevds/uzbek_ner")["train"]

CANON = {"PER": "PERSON", "PERSON": "PERSON", "GPE": "GPE", "LOC": "LOC",
         "ORG": "ORG", "FAC": "FAC", "FACILITY": "FAC", "DATE": "DATE"}
JUNK = {"JCH-2022", "RASUM", "MISC"}

rows_with, total_strings = Counter(), Counter()
for row in raw:
    seen = set()
    for cat, vals in row["ner"].items():
        if vals is None or cat in JUNK:
            continue
        name = CANON.get(cat, cat)
        clean = [v for v in vals if isinstance(v, str) and v.strip()]
        if clean:
            total_strings[name] += len(clean)
            if name not in seen:
                rows_with[name] += 1
                seen.add(name)

tbl = pd.DataFrame({"rows_with_category": pd.Series(rows_with),
                    "total_entity_strings": pd.Series(total_strings)}).fillna(0).astype(int)
tbl["pct_of_rows"] = (tbl["rows_with_category"] / len(raw) * 100).round(1)
tbl = tbl.sort_values("total_entity_strings", ascending=False)
tbl.to_csv(f"{BASE}/phase1_category_frequency.csv")
print(tbl.to_string()); print("\nwritten to Drive")

README.md:   0%|          | 0.00/3.05k [00:00<?, ?B/s]

uzbek_ner.json:   0%|          | 0.00/24.7M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/19609 [00:00<?, ? examples/s]

              rows_with_category  total_entity_strings  pct_of_rows
GPE                        19503                 50036         99.5
LOC                        18898                 38751         96.4
ORG                        18141                 28244         92.5
PERSON                     11996                 25044         61.2
DATE                        6065                 10932         30.9
MONEY                       2200                  5699         11.2
FAC                         2811                  3674         14.3
EVENT                       2676                  3403         13.6
PRODUCT                     1073                  2397          5.5
PERCENT                      396                  1083          2.0
QUANTITY                     296                  1075          1.5
TIME                         509                   933          2.6
CARDINAL                     109                   455          0.6
NORP                         309                